# QuerySense - Experiments Walkthrough

Reproducible notebook version of the QuerySense pipeline. Running this top to
bottom regenerates every number reported in `results/metrics.json`.

**Requirements:** `pip install -r requirements.txt`. The first semantic cell
downloads `sentence-transformers/all-MiniLM-L6-v2` (~90 MB) once and caches it.

**Note on the data:** `data/faqs.csv` is synthetic, illustrative demonstration
data written for this capstone. It does not describe the official policies of
any real institution.

In [1]:
import sys
from pathlib import Path

# Allow `import src...` whether the notebook is run from QuerySense/ or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

pd.set_option("display.max_colwidth", 90)
print("project root:", PROJECT_ROOT)

project root: C:\Users\arjun\OneDrive\Desktop\MUJ-DS-2430010346\QuerySense


## 1. Load and validate the data

Both files ship with the repository, so there is nothing to download
manually. `load_faqs` validates the schema, rejects duplicate ids and empty
fields, and builds the searchable text used for indexing.

In [2]:
from src.data_loader import (
    category_distribution,
    knowledge_base_fingerprint,
    load_evaluation_queries,
    load_faqs,
)

faqs = load_faqs("data/faqs.csv")
queries = load_evaluation_queries("data/evaluation_queries.csv")

print(f"FAQ records : {len(faqs)}")
print(f"Categories  : {faqs['category'].nunique()}")
print(f"Queries     : {len(queries)}")
print(f"Fingerprint : {knowledge_base_fingerprint(faqs)}")
print()
display(category_distribution(faqs))

FAQ records : 150
Categories  : 7
Queries     : 40
Fingerprint : f343a32a25c07c3b



,category,records
0,Fees Scholarships & Administration,24
1,Hostel & Campus Facilities,22
2,Registration & Examinations,22
3,Student Technical Support,22
4,Admissions & Enrollment,20
5,Attendance & Academic Policy,20
6,Library & Learning Resources,20


In [3]:
print("Query type distribution:")
print(queries["query_type"].value_counts().to_string())
print()
display(queries.head(5)[["query_id", "query", "relevant_ids", "query_type"]])

Query type distribution:
query_type
paraphrase       22
combined          9
keyword-heavy     5
short             4



,query_id,query,relevant_ids,query_type
0,Q01,I am not sure whether I am allowed to sit for my end semester papers,REG-005|REG-006,paraphrase
1,Q02,How much of my classes do I need to attend to be allowed to write exams?,REG-006|ATT-001,paraphrase
2,Q03,steps to get into the university as a new student,ADM-001|ADM-009,short
3,Q04,papers I need to submit with my application form,ADM-002,keyword-heavy
4,Q05,"my attendance is very low, what will happen to me?",ATT-003|ATT-004,paraphrase


## 2. Build both retrievers

Both models index the same records. The searchable text is
question + keywords + category - answers are deliberately excluded so long
answers cannot dominate the similarity score.

Model loading and index construction are timed separately from query latency.

In [4]:
from src.tfidf_retriever import TFIDFRetriever
from src.semantic_retriever import SemanticRetriever

doc_ids = faqs["id"].tolist()
texts = faqs["search_text"].tolist()

tfidf = TFIDFRetriever(doc_ids, texts).fit()
print(f"TF-IDF  : vocab={tfidf.vocabulary_size:>6}  build={tfidf.build_seconds:.3f}s")

TF-IDF  : vocab=  2144  build=0.041s


In [5]:
semantic = SemanticRetriever(doc_ids, texts).fit()
print(f"Semantic: dim={semantic.embedding_dim}  load={semantic.model_load_seconds:.2f}s  "
      f"embed={semantic.build_seconds:.2f}s")

C:\Users\arjun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6081.78it/s]

Semantic: dim=384  load=9.94s  embed=0.61s


## 3. Where the two models disagree

Query: **"money back if I leave the course before it starts"**

The FAQ wording is *"What is the refund policy if I withdraw my admission
before classes start?"* - almost no shared vocabulary.

In [6]:
QUERY = "money back if I leave the course before it starts"
looking_up = dict(zip(faqs["id"], faqs["question"]))

rows = []
for model_name, retriever in (("tfidf", tfidf), ("semantic", semantic)):
    for hit in retriever.search(QUERY, k=3):
        rows.append({
            "model": model_name,
            "rank": hit.rank,
            "id": hit.doc_id,
            "score": round(hit.score, 4),
            "question": looking_up[hit.doc_id],
        })

display(pd.DataFrame(rows))

,model,rank,id,score,question
0,tfidf,1,ATT-005,0.2039,Do I need to inform the department before taking leave?
1,tfidf,2,ATT-011,0.1602,What is the procedure for a medical leave certificate?
2,tfidf,3,ATT-006,0.1560,How do I apply for a leave of absence for a semester?
3,semantic,1,ADM-018,0.5495,What is the refund policy if I withdraw my admission before classes start?
4,semantic,2,REG-022,0.5095,Can I withdraw from a course after the drop deadline?
5,semantic,3,REG-003,0.4376,How do I add or drop a course after registration?


TF-IDF anchors on the token **"leave"** and returns leave-of-absence
attendance records. The semantic model returns the actual refund policy at
rank 1. The two result sets are disjoint.

In [7]:
tfidf_ids = set(tfidf.search_ids(QUERY, k=3))
semantic_ids = set(semantic.search_ids(QUERY, k=3))
print("TF-IDF only :", sorted(tfidf_ids - semantic_ids))
print("Semantic only:", sorted(semantic_ids - tfidf_ids))
print("Shared      :", sorted(tfidf_ids & semantic_ids) or "none")
print()
print("Relevant per ground truth: ADM-018, FEE-015")

TF-IDF only : ['ATT-005', 'ATT-006', 'ATT-011']
Semantic only: ['ADM-018', 'REG-003', 'REG-022']
Shared      : none

Relevant per ground truth: ADM-018, FEE-015


## 4. Full evaluation on the 40 labeled queries

Both models run over the identical knowledge base and the identical queries.
Latency is measured in a steady-state pass after a warm-up, with model loading
and index construction excluded.

In [8]:
from src.evaluation import comparison_table, evaluate_all

results = evaluate_all({"tfidf": tfidf, "semantic": semantic}, queries, k=3)
table = comparison_table(results, k=3)
display(table)

,Model,Precision@3,Recall@3,MRR,MAP@3,Mean latency (ms),Median latency (ms),Queries evaluated
0,tfidf,0.2833,0.6625,0.6542,0.5687,0.717,0.657,40
1,semantic,0.3500,0.8250,0.8583,0.7583,11.935,11.109,40


In [9]:
for name, result in results.items():
    s = result["summary"]
    print(f"{name:<9} P@3={s['precision_at_k']:.4f}  R@3={s['recall_at_k']:.4f}  "
          f"MRR={s['mrr']:.4f}  MAP@3={s['map_at_k']:.4f}  "
          f"median={s['median_latency_ms']:.2f}ms  "
          f"no-relevant-retrieved={s['queries_with_no_relevant_retrieved']}/{s['evaluated_queries']}")

tfidf     P@3=0.2833  R@3=0.6625  MRR=0.6542  MAP@3=0.5687  median=0.66ms  no-relevant-retrieved=9/40
semantic  P@3=0.3500  R@3=0.8250  MRR=0.8583  MAP@3=0.7583  median=11.11ms  no-relevant-retrieved=3/40


### Metric comparison chart

Scores are macro-averaged over the 40 queries. Precision@3 uses k as the
denominator, so a query with a single relevant record caps out below 1.0 by
construction.

In [10]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"

metrics = ["Precision@3", "Recall@3", "MRR"]
colors = {"tfidf": "#94a3b8", "semantic": "#4f46e5"}
fig = go.Figure()
for row in table.to_dict("records"):
    fig.add_trace(go.Bar(
        name=row["Model"],
        x=metrics,
        y=[row[m] for m in metrics],
        marker_color=colors[row["Model"]],
        text=[f"{row[m]:.3f}" for m in metrics],
        textposition="outside",
    ))
fig.update_layout(barmode="group", yaxis=dict(range=[0, 1], title="Macro-averaged score (k=3)"),
                  height=400, plot_bgcolor="white", paper_bgcolor="white",
                  legend=dict(orientation="h", y=1.02))
fig

## 5. Qualitative cases

Every query is classified by how the two models behave: agreement, ranking
divergence, single-model success, or joint failure.

In [11]:
from src.evaluation import qualitative_examples

qualitative = qualitative_examples(results, faqs, k=3)
print(qualitative["case_label"].value_counts().to_string())

case_label
Both models succeed (same first relevant rank)    19
Both succeed, ranking differs                     11
Only semantic retrieval succeeds                   7
Both models fail to retrieve a relevant record     2
Only TF-IDF succeeds                               1


In [12]:
cols = ["case_type", "query", "relevant_ids", "tfidf_top_ids", "semantic_top_ids",
        "tfidf_first_relevant_rank", "semantic_first_relevant_rank"]
display(qualitative[qualitative["case_type"] == "semantic_only"][cols].head(7))

,case_type,query,relevant_ids,tfidf_top_ids,semantic_top_ids,tfidf_first_relevant_rank,semantic_first_relevant_rank
30,semantic_only,How much of my classes do I need to attend to be allowed to write exams?,REG-006|ATT-001,ADM-018|ATT-002|FEE-007,REG-004|REG-006|REG-005,0,2
31,semantic_only,papers I need to submit with my application form,ADM-002,LIB-012|ADM-007|ADM-001,ADM-002|ADM-001|FEE-023,0,1
32,semantic_only,electric kettle in my hostel room is it allowed,HOS-012,HOS-006|HOS-003|HOS-002,HOS-012|HOS-005|HOS-021,0,1
33,semantic_only,wireless keeps dropping in the academic block,TEC-004,TEC-003|ATT-013|ATT-020,TEC-004|TEC-008|TEC-003,0,1
34,semantic_only,money back if I leave the course before it starts,ADM-018|FEE-015,ATT-005|ATT-011|ATT-006,ADM-018|REG-022|REG-003,0,1
35,semantic_only,"my exam paper marks look wrong, can I complain",REG-011,FEE-020|REG-010|REG-009,REG-021|REG-010|REG-011,0,3
36,semantic_only,installment option for paying admission charges,FEE-005,ADM-017|HOS-002|ADM-001,FEE-005|ADM-006|FEE-021,0,1


In [13]:
print("Cases where BOTH models fail (honest failure analysis):")
for row in qualitative[qualitative["case_type"] == "both_fail"].itertuples():
    print(f"  {row.query_id}: {row.query}")
    print(f"     relevant={row.relevant_ids}  tfidf={row.tfidf_top_ids}  semantic={row.semantic_top_ids}")

Cases where BOTH models fail (honest failure analysis):
  Q20: how many days can I keep a library book
     relevant=LIB-003  tfidf=LIB-007|LIB-011|LIB-009  semantic=LIB-009|LIB-007|LIB-002
  Q25: can I stay somewhere on campus instead of travelling daily
     relevant=HOS-001|HOS-002  tfidf=HOS-005|HOS-017|HOS-018  semantic=HOS-005|HOS-017|HOS-021


## 6. Conclusions

1. **Semantic retrieval wins on paraphrase handling.** MRR 0.858 vs 0.654 and
   Recall@3 0.825 vs 0.663, with retrieval failures dropping from 9/40 to 3/40.
2. **The gain is not universal.** One query was won by TF-IDF and two were
   missed by both models.
3. **The cost is latency.** Roughly 9x the median query latency plus a one-time
   model load - negligible at this scale, significant at millions of records.
4. **Similarity is not correctness.** A high cosine score means the texts are
   close, not that the retrieved record answers the question.

## 7. Persist the artifacts

Writes `results/metrics.json`, `results/evaluation_results.csv` and
`results/qualitative_examples.csv` - the same files produced by
`python -m src.evaluation`.

In [14]:
from src.evaluation import save_results

paths = save_results(results, faqs, queries, k=3,
                     output_dir=PROJECT_ROOT / "results", qualitative=qualitative)
for label, path in paths.items():
    print(f"{label:<12} {path}")

metrics      C:\Users\arjun\OneDrive\Desktop\MUJ-DS-2430010346\QuerySense\results\metrics.json
results      C:\Users\arjun\OneDrive\Desktop\MUJ-DS-2430010346\QuerySense\results\evaluation_results.csv
qualitative  C:\Users\arjun\OneDrive\Desktop\MUJ-DS-2430010346\QuerySense\results\qualitative_examples.csv
